[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C28_Frontier_Diffusion_Course/05_consistency/05_consistency.ipynb)

# 05 · 一致性模型（用 numpy 玩具复现）

目标：用纯 numpy 把 **consistency model 的核心**从零搭出来：① 已知闭式的直线 ODE 轨迹；② **边界条件**由 skip 参数化硬保证；③ **自洽损失**作训练信号；④ **单步 / 多步采样**。

路线：直线轨迹 + 已知终点 → skip 参数化 + 边界恒等 → 精确一致性函数 → 自洽损失 → 训一个小 F → 单步采样 → 多步精修 → ✏️ 练习 → 📖 答案 → 🧪 真实 NFE/LCM 胶囊。

> 心智模型：**CM = 学「从轨迹任意点直达终点」的函数 f；边界条件钉住 t=0、自洽性把终点沿轨迹传播**。单步一次前向出图，多步精修提质。

> 我们借用模块 03 的直线轨迹 `x_t=(1-t)·x1+t·x0`（数据 x1、噪声 x0），它的「终点」闭式可得，便于逐项验证。

## 1 · 已知闭式的 ODE 轨迹（含确定性 噪声↔数据 配对）

CM 之所以能**单步**生成，根基是 probability flow ODE 给出噪声↔数据的**确定性双射**：每个噪声唯一对应一个数据点。

玩具里我们直接造这个双射：数据由噪声经一个固定映射 `g` 得到，`x1=g(x0)`（这样「从噪声预测数据」是良定义的可学函数）。轨迹用直线（模块 03 路径）：

$$x_t=(1-t)\,x_1+t\,x_0,\quad t\in[0,1],$$

`x1`=数据（终点，t=0 端）、`x0`=噪声（t=1 端，且 `x1=g(x0)`）。轨迹「终点」闭式可解：`x1=(x_t-t·x0)/(1-t)`（t<1）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def data_map(x0):
    '''固定的「噪声->数据」映射 g：把高斯噪声变形成 2D 双团状数据。
       关键：数据是噪声的确定性、平滑函数 -> 单步生成 f(noise,1)=g(noise) 才良定义且可学。
       用 tanh(scale·x) 近似「按第一维符号分两团」，但保持平滑(线性+tanh 特征可拟合)。'''
    cluster = 1.5 * np.tanh(3.0 * x0[:, [0]])      # ≈ ±1.5 两团(平滑)
    return np.concatenate([cluster, 0.2 * x0[:, [1]]], axis=1)

def sample_noise(n, rng):
    return rng.standard_normal((n, 2))

def sample_pair(n, rng):
    '''返回配对的 (数据 x1, 噪声 x0)，满足 x1=g(x0)。'''
    x0 = sample_noise(n, rng)
    return data_map(x0), x0

def traj_point(x1, x0, t):
    '''直线轨迹上 t 时刻的点：x_t=(1-t)x1+t x0。'''
    return (1 - t) * x1 + t * x0

def traj_origin(x_t, x0, t):
    '''闭式求轨迹终点(数据 x1)：x1=(x_t - t·x0)/(1-t)。t<1。'''
    return (x_t - t * x0) / (1 - t)

x1, x0 = sample_pair(1000, rng)      # 配对的 数据/噪声
# 数据确是双团：第一维多数点聚在 ±1.5 附近(|x|>0.9)
assert np.mean(np.abs(x1[:, 0]) > 0.9) > 0.7, '数据应大致是 ±1.5 双团'
# 取轨迹上某 t 的点，再闭式还原终点，应精确等于 x1
for t in [0.0, 0.3, 0.7, 0.95]:
    xt = traj_point(x1, x0, t)
    x1_back = traj_origin(xt, x0, t)
    err = np.abs(x1_back - x1).max()
    print(f't={t:.2f}: 闭式还原终点误差 = {err:.2e}')
    assert err < 1e-9, '闭式还原终点应精确'
print('✅ 直线轨迹 + 确定性配对：任意 t 的点都能精确还原终点 x1=g(x0)')

## 2 · skip 参数化：边界条件硬保证

CM 要求 **f(x,0)=x**（t=0 无噪声、f 是恒等）。不靠 loss，靠参数化硬保证：

$$f(x,t)=c_{\text{skip}}(t)\,x+c_{\text{out}}(t)\,F(x,t),\quad c_{\text{skip}}(0)=1,\;c_{\text{out}}(0)=0.$$

于是 `f(x,0)=1·x+0·F=x`，**不论 F 输出什么**。验证：用一个随机乱来的 F，f 在 t=0 仍严格等于 x。

In [ ]:
def c_skip(t):
    '''t=0 -> 1, t=1 -> 0。这里用 (1-t) 这个最简形式。'''
    return 1.0 - t

def c_out(t):
    '''t=0 -> 0。用 t。'''
    return t

def make_f(F):
    '''把任意网络 F 包成满足边界条件的一致性函数 f。'''
    def f(x, t):
        return c_skip(t) * x + c_out(t) * F(x, t)
    return f

# 一个「乱来」的 F（内容无所谓，关键看 t=0 被 c_out=0 抹掉）
Wrand = rng.standard_normal((2, 2)) * 5.0
F_rand = lambda x, t: np.tanh(x @ Wrand) * 100.0   # 故意巨大、非线性
f_rand = make_f(F_rand)

x = rng.standard_normal((50, 2))
# 边界：t=0 时 f(x,0) 必须严格 == x
f0 = f_rand(x, 0.0)
print('边界检验 |f(x,0)-x| =', np.abs(f0 - x).max())
assert np.allclose(f0, x, atol=1e-12), 'skip 参数化必须使 f(x,0)=x 严格成立'
# t>0 时 f 会偏离 x（F 起作用）
assert not np.allclose(f_rand(x, 0.5), x), 't>0 时 F 应起作用'
assert c_skip(0.0) == 1.0 and c_out(0.0) == 0.0
print('✅ 边界条件由参数化硬保证：f(x,0)=x 严格成立，与 F 无关（adaLN-zero 同款智慧）')

## 3 · 精确一致性函数：把任意点映到终点

先构造一个**精确的** f（用闭式终点），它演示了 CM 想学的目标：把同一轨迹上任意点映到同一个终点 x1。

精确 f 就是 `traj_origin`（已知 x0）。但真实 CM 不知道 x0，要让 f 只凭 (x_t, t) 输出终点。这里先用闭式版验证**自洽性**这个目标性质。

In [ ]:
def f_exact(x_t, t, x0):
    '''精确一致性函数：闭式把轨迹点映回终点。真实 CM 要学它(不给 x0)。'''
    if t < 1e-9:
        return x_t                 # 边界：t=0 即终点
    return traj_origin(x_t, x0, t)

# 自洽性：同一轨迹上不同 t 的点，f_exact 应给出同一个 x1
x1, x0 = sample_pair(500, rng)
out_at = {}
for t in [0.2, 0.5, 0.8]:
    xt = traj_point(x1, x0, t)
    out_at[t] = f_exact(xt, t, x0)
# 三个时刻的输出应彼此一致(都=x1)
assert np.allclose(out_at[0.2], out_at[0.5], atol=1e-9)
assert np.allclose(out_at[0.5], out_at[0.8], atol=1e-9)
assert np.allclose(out_at[0.5], x1, atol=1e-9)
print('f_exact 在 t=0.2/0.5/0.8 的输出两两一致，且都=x1 ✅')
print('✅ 这就是 CM 的目标：同一轨迹任意点 -> 同一终点(自洽性)')

## 4 · 自洽损失：相邻时刻输出之差

真实 CM 不知道终点，靠**自洽损失**学：同一轨迹相邻两点 `t_n, t_{n+1}` 的 f 输出应一致。

$$\mathcal{L}=\mathbb{E}\,\big\|f(x_{t_{n+1}},t_{n+1})-f(x_{t_n},t_n)\big\|^2.$$

验证：对**精确** f，自洽损失=0（完美自洽）；对一个**乱来**的 f，自洽损失>0。

In [ ]:
def consistency_loss(f_fn, x1, x0, t_lo, t_hi):
    '''同一轨迹上相邻 (t_lo, t_hi) 两点，f 输出之差的均方。'''
    x_hi = traj_point(x1, x0, t_hi)
    x_lo = traj_point(x1, x0, t_lo)
    o_hi = f_fn(x_hi, t_hi)
    o_lo = f_fn(x_lo, t_lo)
    return np.mean((o_hi - o_lo) ** 2)

x1, x0 = sample_pair(500, rng)
# 精确 f（闭包固定这批 x0）-> 自洽损失应=0
f_ex = lambda x, t: f_exact(x, t, x0)
L_exact = consistency_loss(f_ex, x1, x0, 0.4, 0.5)
print(f'精确 f 的自洽损失 = {L_exact:.2e}')
assert L_exact < 1e-12, '精确 f 应完美自洽(损失≈0)'
# 乱来的 f -> 自洽损失明显>0
L_bad = consistency_loss(f_rand, x1, x0, 0.4, 0.5)
print(f'乱来 f 的自洽损失 = {L_bad:.3f}')
assert L_bad > 1e-3, '不自洽的 f 损失应明显>0'
print('✅ 自洽损失：精确 f 为 0、乱来 f >0 —— 它是可优化的训练信号')

## 5 · 训一个小 F：让自洽损失下降

把「学 CM」落实：用最小二乘训一个线性 `F(x,t)`，使包装后的 f 在多对相邻 (t_n,t_{n+1}) 上自洽损失下降。

关键纪律（防平凡解）：用**边界条件**(t=0 钉死) + 把训练目标设为「相邻时刻向闭式终点对齐」。这里用闭式终点当监督(相当于蒸馏的老师信号)，训 F 使 f≈终点。

In [ ]:
# 特征：把 (x_t, t) 编码成特征，回归出「F 的输出」使 f≈x1(=g(x0))
# f = c_skip(t)*x + c_out(t)*F ；要 f≈x1 => F ≈ (x1 - c_skip(t)*x)/c_out(t)  (t>0)
# 因 g 含 tanh，特征里加 tanh(3·x_t) 让线性回归能表达这种非线性
def featurize(x_t, t):
    '''特征：[x_t, tanh(3·x_t), t·x_t, 1, t]（含非线性项以拟合 g）。'''
    n = x_t.shape[0]
    tcol = np.full((n, 1), t)
    return np.concatenate([x_t, np.tanh(3.0 * x_t), t * x_t,
                           np.ones((n, 1)), tcol], axis=1)

# 造训练集：多条轨迹、多个 t(含接近 1 的)，目标是 F 的「理想输出」使 f=x1
x1_tr, x0_tr = sample_pair(6000, rng)
feats, targets = [], []
for t in np.linspace(0.1, 0.999, 16):
    xt = traj_point(x1_tr, x0_tr, t)
    F_ideal = (x1_tr - c_skip(t) * xt) / c_out(t)    # 使 f=x1
    feats.append(featurize(xt, t)); targets.append(F_ideal)
Phi = np.concatenate(feats, 0); Y = np.concatenate(targets, 0)
# 最小二乘解 F 的线性权重
Wf, *_ = np.linalg.lstsq(Phi, Y, rcond=None)
F_learned = lambda x, t: featurize(x, t) @ Wf
f_learned = make_f(F_learned)

# 边界仍严格成立(与 F 无关)
xb = rng.standard_normal((20, 2))
assert np.allclose(f_learned(xb, 0.0), xb, atol=1e-12)
# 学到的 f 在轨迹点上接近真终点(蒸馏意义)
x1_te, x0_te = sample_pair(500, rng)
err_mid = np.mean((f_learned(traj_point(x1_te, x0_te, 0.5), 0.5) - x1_te) ** 2)
print(f'学到的 f 在 t=0.5 处预测终点的 MSE = {err_mid:.4f}')
assert err_mid < 0.2, '训练后 f 应较准地指向终点'
# 自洽损失：学到的 f 远小于乱来的 f
L_learned = consistency_loss(f_learned, x1_te, x0_te, 0.4, 0.5)
L_rand = consistency_loss(f_rand, x1_te, x0_te, 0.4, 0.5)
print(f'学到的 f 自洽损失 = {L_learned:.4f}   乱来 f = {L_rand:.4f}')
assert L_learned < L_rand, '训练后自洽损失应显著下降'
print('✅ 训一个小 F：f 学会沿轨迹一致地指向终点 g(x0)，自洽损失大幅下降')

## 6 · 单步与多步采样

**单步**：从噪声 `x_T` 一次 `f(x_T, 1)` 出图。**多步**：交替「加噪回中途 → 再跳回终点」精修。

验证：① 单步生成落在数据附近（双团结构）；② 多步精修让「到最近数据团中心的距离」下降。

In [ ]:
def sample_single_step(f_fn, n, rng):
    x_T = rng.standard_normal((n, 2))      # 噪声(t=1 端)
    return f_fn(x_T, 1.0)                   # 一次前向直达终点

def sample_multi_step(f_fn, n, taus, rng):
    '''多步精修：先单步，再对每个中间 tau 做 加噪->跳回。'''
    x = sample_single_step(f_fn, n, rng)
    for tau in taus:
        x = x + tau * rng.standard_normal(x.shape)   # 加噪回噪声级别 tau
        x = f_fn(x, tau)                              # 再跳回终点
    return x

centers = np.array([[-1.5, 0.0], [1.5, 0.0]])
def dist_to_data(x):
    '''到最近数据团中心的平均距离(越小越贴近数据)。'''
    d = np.minimum(np.linalg.norm(x - centers[0], axis=1),
                   np.linalg.norm(x - centers[1], axis=1))
    return d.mean()

x_1step = sample_single_step(f_learned, 2000, np.random.default_rng(1))
x_4step = sample_multi_step(f_learned, 2000, [0.6, 0.4, 0.2], np.random.default_rng(1))
d1 = dist_to_data(x_1step); d4 = dist_to_data(x_4step)
# 噪声基线：纯噪声到数据中心的距离
d_noise = dist_to_data(np.random.default_rng(1).standard_normal((2000, 2)))
print(f'纯噪声  到数据距离 = {d_noise:.3f}')
print(f'单步生成 到数据距离 = {d1:.3f}')
print(f'4步精修 到数据距离 = {d4:.3f}')
assert d1 < d_noise, '单步生成应远比噪声贴近数据'
assert d4 <= d1 + 1e-9, '多步精修不应更差(通常更好)'
# 生成落在两团附近：x 坐标应聚到 ±1.5 附近
frac_near = np.mean(np.min(np.abs(x_1step[:, [0]] - centers[:, 0]), axis=1) < 0.8)
assert frac_near > 0.6, '多数生成点应落在某个数据团附近'
print('✅ 单步即可生成(落在数据附近)；多步精修进一步贴近数据分布')

---
## ✏️ 练习 1：自洽损失

实现 `self_consistency_loss(f_fn, x1, x0, t_lo, t_hi)`：同一轨迹相邻两点的 f 输出之**均方差**。

（即第 4 节 `consistency_loss` 的逻辑：在 t_lo、t_hi 各取轨迹点，算 f 输出差的均方。）

In [ ]:
def self_consistency_loss(f_fn, x1, x0, t_lo, t_hi):
    # TODO: x_hi=traj_point(x1,x0,t_hi); x_lo=traj_point(x1,x0,t_lo)
    #       返回 mean((f(x_hi,t_hi)-f(x_lo,t_lo))**2)
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
x1, x0 = sample_pair(400, rng)
f_ex = lambda x, t: f_exact(x, t, x0)
# 精确 f -> 损失≈0
assert self_consistency_loss(f_ex, x1, x0, 0.3, 0.4) < 1e-12
# 乱来 f -> 损失>0
assert self_consistency_loss(f_rand, x1, x0, 0.3, 0.4) > 1e-3
# 与参考实现一致
assert np.isclose(self_consistency_loss(f_rand, x1, x0, 0.3, 0.5),
                  consistency_loss(f_rand, x1, x0, 0.3, 0.5))
print('✅ 练习 1 通过：自洽损失正确(精确 f 为 0、乱来 f >0)')

## ✏️ 练习 2：边界条件参数化

实现一对预条件系数 `my_c_skip(t)`、`my_c_out(t)`，满足边界 `c_skip(0)=1, c_out(0)=0`，并用它们包装 `make_f_custom(F, c_skip, c_out)` 使 `f(x,0)=x` 严格成立。

In [ ]:
def my_c_skip(t):
    # TODO: 任一满足 c_skip(0)=1 的函数，例如 1-t 或 (1-t)**2
    raise NotImplementedError

def my_c_out(t):
    # TODO: 任一满足 c_out(0)=0 的函数，例如 t
    raise NotImplementedError

def make_f_custom(F, cs, co):
    # TODO: 返回 f(x,t)=cs(t)*x+co(t)*F(x,t)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert abs(my_c_skip(0.0) - 1.0) < 1e-12, 'c_skip(0) 必须=1'
assert abs(my_c_out(0.0) - 0.0) < 1e-12, 'c_out(0) 必须=0'
f_custom = make_f_custom(F_rand, my_c_skip, my_c_out)
xb = rng.standard_normal((30, 2))
# 边界严格成立(与 F 无关)
assert np.allclose(f_custom(xb, 0.0), xb, atol=1e-12), 'f(x,0)=x 必须严格成立'
# t>0 时 F 起作用
assert not np.allclose(f_custom(xb, 0.5), xb)
print('✅ 练习 2 通过：自定义预条件满足边界、参数化硬保证 f(x,0)=x')

## ✏️ 练习 3：多步采样

实现 `multi_step(f_fn, x_init, taus, rng)`：从 `x_init`（已是一次估计）开始，对每个 `tau` 做「加噪 `+tau*z` → 跳回 `f(·,tau)`」，返回精修结果。

（即第 6 节 `sample_multi_step` 的精修循环，但接收已有的 `x_init`。）

In [ ]:
def multi_step(f_fn, x_init, taus, rng):
    # TODO: x=x_init.copy(); for tau in taus: x=x+tau*噪声; x=f_fn(x,tau); 返回 x
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
g = np.random.default_rng(2)
x_init = sample_single_step(f_learned, 1500, g)
x_ref = multi_step(f_learned, x_init, [0.5, 0.3], np.random.default_rng(5))
# taus 为空 -> 原样返回
assert np.allclose(multi_step(f_learned, x_init, [], g), x_init)
# 精修后到数据距离不增(用同一随机种子比较的稳健性: 用平均距离)
d_init = dist_to_data(x_init); d_ref = dist_to_data(x_ref)
print(f'精修前距离={d_init:.3f}  精修后距离={d_ref:.3f}')
assert d_ref <= d_init + 0.05, '多步精修(均值意义)不应明显变差'
assert x_ref.shape == x_init.shape
print('✅ 练习 3 通过：多步精修循环正确，可在质量-步数间滑动')

## ✏️ 练习 4：蒸馏的相邻点构造

一致性蒸馏(CD)用老师沿 ODE 走一步造相邻点。这里给定老师能闭式还原终点，实现 `make_adjacent_pair(x1, x0, t_hi, dt)`：返回 `(x_{t_hi}, x_{t_lo})`，其中 `t_lo=t_hi-dt`，两点在**同一条轨迹**上。

In [ ]:
def make_adjacent_pair(x1, x0, t_hi, dt):
    # TODO: t_lo=t_hi-dt; 返回 (traj_point(x1,x0,t_hi), traj_point(x1,x0,t_lo))
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
x1, x0 = sample_pair(300, rng)
x_hi, x_lo = make_adjacent_pair(x1, x0, 0.6, 0.1)
# 两点确在同一轨迹上：闭式还原的终点应相同
o_hi = traj_origin(x_hi, x0, 0.6)
o_lo = traj_origin(x_lo, x0, 0.5)
assert np.allclose(o_hi, o_lo, atol=1e-9), '相邻点应在同一轨迹(还原同一终点)'
assert np.allclose(o_hi, x1, atol=1e-9)
# 两点确实不同(相邻但不重合)
assert not np.allclose(x_hi, x_lo)
print('✅ 练习 4 通过：蒸馏的相邻点构造正确(同轨迹、可作自洽训练对)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def self_consistency_loss(f_fn, x1, x0, t_lo, t_hi):
    x_hi = traj_point(x1, x0, t_hi)
    x_lo = traj_point(x1, x0, t_lo)
    return np.mean((f_fn(x_hi, t_hi) - f_fn(x_lo, t_lo)) ** 2)

In [ ]:
# 练习 2 参考答案
def my_c_skip(t):
    return (1 - t) ** 2          # c_skip(0)=1
def my_c_out(t):
    return t                      # c_out(0)=0
def make_f_custom(F, cs, co):
    def f(x, t):
        return cs(t) * x + co(t) * F(x, t)
    return f

In [ ]:
# 练习 3 参考答案
def multi_step(f_fn, x_init, taus, rng):
    x = x_init.copy()
    for tau in taus:
        x = x + tau * rng.standard_normal(x.shape)
        x = f_fn(x, tau)
    return x

In [ ]:
# 练习 4 参考答案
def make_adjacent_pair(x1, x0, t_hi, dt):
    t_lo = t_hi - dt
    return traj_point(x1, x0, t_hi), traj_point(x1, x0, t_lo)

---
## 🧪 真实数据胶囊：NFE 与少步生成的加速账

用**真实模型**的步数(NFE，number of function evaluations)算少步生成省了多少。扩散采样器要几十步，CM/LCM/Turbo 只要 1~4 步。下面是公开的典型 NFE 配置。

算每个少步模型相对标准扩散(以 DDIM 50 步为基线)的**加速比**(NFE 之比)。

In [ ]:
# 真实模型的典型采样 NFE(公开资料；CFG 会让每步×2，这里记为基础前向数)
NFE_CONFIGS = {
    'DDPM (原版)':       1000,
    'DDIM (常用)':        50,
    'DPM-Solver++':       20,
    'LCM (4步)':           4,
    'SDXL-Turbo (1-4步)':  1,
}
baseline = NFE_CONFIGS['DDIM (常用)']
print(f"{'模型/采样器':<22}{'NFE':>6}{'相对DDIM50加速':>16}")
for name, nfe in NFE_CONFIGS.items():
    speedup = baseline / nfe
    print(f'{name:<22}{nfe:>6}{speedup:>14.1f}x')
# LCM 4 步相对 DDIM 50 步 -> 12.5x；Turbo 1 步 -> 50x
print('\n观察：从 DDIM 50 步到 LCM 4 步约 12.5x、到 Turbo 1 步 50x 加速')
print('     这就是「质量略降、速度暴涨」的少步生成 —— 实时交互的关键')

**🧪 胶囊练习**：实现 `speedup_vs(nfe, baseline_nfe)` 返回加速比 `baseline_nfe/nfe`；并算：若标准管线用 DDIM 50 步 + CFG(每步跑 2 次网络=100 次前向)，LCM 蒸馏掉 CFG 后 4 步(每步 1 次=4 次前向)，**端到端**前向次数加速多少倍？

In [ ]:
def speedup_vs(nfe, baseline_nfe):
    # TODO: 返回 baseline_nfe / nfe
    raise NotImplementedError

def end_to_end_forward_speedup(ddim_steps, lcm_steps, ddim_cfg=True, lcm_cfg=False):
    # TODO: 基线前向数 = ddim_steps*(2 if ddim_cfg else 1)
    #       LCM 前向数 = lcm_steps*(2 if lcm_cfg else 1)
    #       返回 基线/LCM
    raise NotImplementedError

In [ ]:
# 自测
assert abs(speedup_vs(4, 50) - 12.5) < 1e-9
assert abs(speedup_vs(1, 50) - 50.0) < 1e-9
# DDIM50+CFG=100 次前向；LCM4 无CFG=4 次前向 -> 25x
s = end_to_end_forward_speedup(50, 4, ddim_cfg=True, lcm_cfg=False)
assert abs(s - 25.0) < 1e-9, 'DDIM50+CFG vs LCM4 应 25x'
print('LCM 蒸馏掉 CFG + 4 步：端到端前向数加速 25x（100 次 -> 4 次）')
print('✅ 胶囊练习通过：少步 + 蒸馏掉 CFG 是双重加速')

In [ ]:
# 📖 胶囊参考答案
def speedup_vs(nfe, baseline_nfe):
    return baseline_nfe / nfe
def end_to_end_forward_speedup(ddim_steps, lcm_steps, ddim_cfg=True, lcm_cfg=False):
    base = ddim_steps * (2 if ddim_cfg else 1)
    lcm = lcm_steps * (2 if lcm_cfg else 1)
    return base / lcm

---
### 小结
- **consistency model = 学「把 ODE 轨迹上任意点直达终点(数据)」的函数 f**，从而单步生成、少步精修。
- **自洽性**：同一轨迹相邻点的 f 输出必须一致；用不回传梯度的目标网络 θ⁻ 防平凡解(常数)。
- **边界条件** `f(x,0)=x` 由 skip 参数化 `f=c_skip·x+c_out·F`(c_skip(0)=1,c_out(0)=0) **硬保证**——与 adaLN-zero 同款智慧。
- **蒸馏(CD)** 用扩散老师造相邻点(稳、主流)；**从零训练(CT)** 不需老师(难)；前身是渐进蒸馏。
- **质量-步数权衡**：单步最快最糙，多步精修提质；CM/LCM 用「快 10~50x」换「质量略降」——实时生成的关键。
- **LCM** = latent 空间 CM + 蒸馏掉 CFG，让 SD 4 步出图；LCM-LoRA 可插到任意现成模型。

🎉 **恭喜：你已走完 C28 全部五个前沿模块**——latent 扩散(01)、DiT(02)、flow matching(03)、CFG(04)、consistency(05)。把它们拼起来，就是一台像 SD3/Flux 那样「在 latent 上、用 DiT 骨干、按 flow 训练、用 CFG 控制、被蒸馏成少步」的现代生成模型。